# Python: Algorithms

## 1. Foundations

### 1.1. General framework
- A problem clearly specifies the valid inputs and the required output.
- An algorithm is a step-by-step method that guarantees the correct output for any valid input.
- A program is the implemented solution: the algorithm realized in code, together with the data structures it uses.

### 1.2. Algorithm analysis
Algorithm analysis is the process of evaluating an algorithm to understand how well it works. It focuses on two main questions: does the algorithm produce the correct output, and how efficiently does it use resources?
- Correctness can be argued using test cases or a mathematical proof. Test cases are fast and practical, but they only check a limited set of inputs and can miss edge cases. A mathematical proof is harder but more general, since it can guarantee correctness for all valid inputs. Common proof techniques include reasoning with recursion (often using induction) or establishing a loop invariant.
- Efficiency is captured by complexity analysis. Time complexity describes how the running time grows as the input size increases, while space complexity describes how much memory the algorithm uses as the input size increases.

#### Big O notation
Time/space complexity is usually described by *worst-case* and *average-case*. Worst-case gives an upper bound on how slow or memory-heavy the algorithm can get on any valid input, while average-case estimates typical performance over a distribution of inputs (when that distribution makes sense). We often summarize growth rates using Big-O and compare common forms. A typical ordering is:

$$O(1) < O(\log n) < O(n) < O(n \log n) < O(n^2) < O(2^n) < O(n!)$$

This comparison helps you predict scalability: moving right gets expensive fast, and exponential/factorial growth becomes infeasible even for moderate $n$. To determine complexity for an algorithm:
- First define the input size $n$ and what counts as a basic operation (e.g., comparisons, swaps, hash lookups).
- Then count how many times key operations run: a single loop is usually $O(n)$, nested loops often give $O(n^2)$, and a loop that halves/doubles the remaining work tends to be $O(\log n)$. Divide-and-conquer patterns often lead to $O(n\log n)$ (like split into halves, do linear work per level). Exponential behavior like $O(2^n)$ typically comes from branching recursion that explores many combinations, while $O(n!)$ shows up when you try all permutations. For recursive algorithms, write a recurrence (e.g., $T(n)=2T(n/2)+O(n)$) and solve it.
- Finally, drop constants and lower-order terms to keep the dominant growth rate, and do the same for memory by tracking extra storage (arrays, recursion stack, auxiliary structures).

### 1.3. Some strategies

#### Recursion vs. iteration
Recursion and iteration are two ways to repeat work:
- Iteration uses loops and updates state variables until a stopping condition is reached.
- Recursion expresses a function calling itself on incremental input until reaching a base case. It's the implemented version of Math induction.

While recursion can feel a bit like a brain teaser at first, it's the most nature way to write algorithms on recursive structures. But iteration is usually more efficient and stable in practice. You can perform *recursion elimination* techniques to convert recursive code to iterative.

:::{admonition} Case: Factorial
:class: tip

To calculate the factorial $n!$ recursively, we must clearly define these:
- The base case (when to stop, what to return): when $n=1$, return $1$.
- Induction logic (how the chain of problems relate): $n!=n\times(n-1)!$ Note that each step must get closer to the base case.

:::

In [2]:
def factorial(n):
    if n == 0 or n == 1:
        return 1
    else:
        return n * factorial(n - 1)

print(factorial(5))

120


#### Reduction
Reduction is a problem-solving strategy where you transform a new problem into another problem that you already know how to solve. After solving the target problem, you map the result back to the original. Reduction is not a data structure or a single algorithm family; it is a meta-technique that helps you reuse existing algorithms and proofs. Common patterns include reducing to sorting (e.g., interval problems), reducing to graph problems (shortest path, MST, max flow), or reducing to a known data structure operation (e.g., connectivity queries reduced to union-find). Reductions are also used to prove hardness by showing that solving your problem would imply solving a known hard problem.

## 2. Data structures
Data structures organize data so algorithms can work efficiently. They’re often grouped into *linear* structures, which store items in a single sequence (such as lists, arrays, stacks, and queues), and *non-linear* structures, which store items in branching or network-like relationships (such as trees, graphs, and sets).

### 2.1. Linear structures

#### Array
The core idea of array (like Python `tuple`) is to store items side-by-side in contiguous memory locations. Because every position is directly addressable, indexing an element and iterating through all elements are fast $O(1)$. But inserting and deleting elements can be costly $O(n)$ since it may require shifting other elements. Arrays are fixed size, so resizing is involves creating a new array and copying over elements, which is also $O(n)$.

That's why dynamic array (like Python `list`) uses doubling capacity to amortize the cost of resizing over multiple insertions. The core idea is to act like it can grow: allocate double capacity than actual used slots, so that insertions are $O(1)$. When running out of space (occasionally happens), create a new array with double size and copy over elements. This way, most insertions are still $O(1)$ amortized, while occasional resizing is $O(n)$.

Arrays are great when you need random access, tight loop over data and operations are mostly read/append.

#### Linked list
The core idea of a linked list is to store items as separate nodes connected by pointers, not in contiguous memory. In a singly linked list, each node points to the next; in a doubly linked list, each node points to both next and previous. Pros and cons:
- Because nodes are not contiguous, random access is slow: getting the i-th element requires traversal from the head, so indexing is $O(n)$. Iteration and searching for a value are also $O(n)$.
- The advantage is insertion/deletion without shifting. If you already have a reference to the position (or you insert/delete at the head), you only change a few pointers, so insertion/deletion can be $O(1)$. Doubly linked lists use more memory but support $O(1)$ deletion of a known node and easy backward movement.

Linked lists are useful when you do frequent insertions/deletions near a current position. For heavy indexing and tight loops, arrays/dynamic arrays are usually better.

#### Abstract types
Stack, queue, and deque are abstract data types: they define allowed operations and the order elements come out, but they do not specify how data is stored. In practice they can be implemented using arrays/dynamic arrays or linked lists.
- A stack is LIFO (last-in, first-out). Its core operations are push, pop, and top, which are typically $O(1)$. Stacks are useful for nested structure and backtracking, such as matching parentheses, undo operations, and depth-first search.
- A queue is FIFO (first-in, first-out). Its core operations are enqueue, dequeue, and front, which are typically $O(1)$. Queues are useful for processing in arrival order, such as breadth-first search, simulations, and task scheduling.
- A deque (double-ended queue) supports pushing and popping at both the front and back, typically $O(1)$. Deques are useful when you need both ends, especially in sliding window algorithms (e.g., maintaining window max/min with a monotonic deque).

### 2.2. Non-linear structures

#### Hash table
A hash table is a data structure that uses a hash function to convert a key into a table position, so lookup/insert/delete are typically $O(1)$ on average. Map and set are abstract interfaces: a map supports key-value operations, and a set supports membership operations on keys. In Python, `dict` is a map and `set` is a set; both are built-in classes implemented using hash-table ideas.

Sometimes different keys land in the same position (collision). Implementations handle this, but too many collisions can degrade performance toward $O(n)$ in the worst case. Similar to dynamic arrays, hash tables occasionally resize when they get too full, so inserts stay $O(1)$ amortized. Use them for fast membership tests, frequency counting, caching, and key-value lookup when you don’t need ordering.

#### Heap
A heap is a data structure designed to quickly keep track of the most important element. It maintains a simple rule: in a min-heap, every parent is smaller than its children (in a max-heap, every parent is larger). This doesn't fully sort the data, but it guarantees the minimum/maximum is always at the top.

A priority queue is the abstract interface: insert items with priorities, and repeatedly remove the item with highest priority (or lowest, depending on convention). A heap is the most common way to implement a priority queue.

Because the heap stays roughly balanced, inserting an item and removing the top element both take $O(\log n)$, while just peeking at the top is $O(1)$. Heaps are great when you repeatedly need the next best candidate, like scheduling tasks by priority, selecting the top-k elements, or running graph algorithms such as Dijkstra's shortest path.

#### Tree
A tree is a data structure for representing hierarchical relationships. It consists of nodes connected by edges, with one root node at the top; every node (except the root) has exactly one parent, and can have zero or more children. Because of this structure, trees are great for modeling things like folder structures, organization charts, and parsed expressions.

Common operations include traversals (visit all nodes) like preorder/inorder/postorder and level-order; these take $O(n)$ because you touch each node once. Many tree algorithms are naturally recursive, but can also be written iteratively using an explicit stack/queue.

A very common special case is the binary search tree, where for each node, all keys in the left subtree are smaller and all keys in the right subtree are larger. This ordering makes search/insert/delete efficient: typically $O(\log n)$ when the tree stays reasonably balanced, but it can degrade to $O(n)$ if the tree becomes a chain.

## 3. Exact methods

:::{mermaid}
:align: center

flowchart TD
  P["Exact method"]
  BF["Brute force"]
  BT["Backtracking"]
  BB["Branch and Bound"]
  DC["Divide and Conquer"]
  DP["Dynamic Programming"]

  P --> BF
  BF --> |feasibility<br>pruning| BT
  BT --> |optimality<br>pruning| BB
  P --> DC
  DC --> |memoization<br>tabulation| DP

:::

### 3.1. Brute force
Brute force is the simplest problem-solving approach: it tries all possible options util you find the answer (or the best answer). While often inefficient, brute force is easy to implement and guarantees finding the optimal solution. It's useful for small input sizes and you don't want to overcomplicate things.

:::{admonition} Case: Two sum
:class: tip

- Input: an array of integers `array` and an integer `target`.
- Output: The indices $i,j$ of the two elements that add up to `target`. If no such pair exists, return `None`.

Implementation using brute force involves a nested loop (two levels) to check all pairs. The time complexity is $O(n^2)$.

:::

In [ ]:
def two_sum(array, target):
    n = len(array)
    for i in range(n):
        for j in range(i + 1, n):
            if array[i] + array[j] == target:
                return i, j

two_sum([4, 3, 8, 2, 7, 11, 15], 9)

(3, 4)

### 3.2. Divide and conquer
Divide and conquer solves a problem by splitting it into smaller subproblems of the same type. It involves three main steps:
- Divide: break the input into smaller parts
- Conquer: solve each part (usually recursively) until the parts are small enough to solve directly
- Combine: merge the sub-results into the final answer

It works best when the subproblems are mostly independent and combining results is efficient.

:::{admonition} Case: Binary search
:class: tip

- Input: a sorted array of integers `array` and an integer `target`.
- Output: The index of `target` in `array`, or `-1` if not found.

The core idea of binary search is to repeatedly discard half of the array which is guaranteed not to contain the target. This is done by comparing the target with the middle element of the current array, which requires the array to be sorted. The time complexity of binary search is $O(\log n)$.

:::

In [ ]:
def binary_search_iterative(array, target):
    left, right = 0, len(array) - 1
    while left <= right:
        mid = (left + right) // 2
        if array[mid] == target:
            return mid
        elif array[mid] < target:
            left = mid + 1
        else:
            right = mid - 1
    return -1

In [ ]:
def binary_search_recursive(array, target):
    
    def helper(left, right):
        if left > right:
            return -1
        mid = (left + right) // 2
        if array[mid] == target:
            return mid
        if array[mid] < target:
            return helper(mid + 1, right)
        return helper(left, mid - 1)

    return helper(0, len(array) - 1)

:::{admonition} Case: Exponentiation by squaring
:class: tip

$b^n$ can be computed efficiently using the idea of divide and conquer with the time complexity $O(\log n)$:

$$\begin{aligned}
n=2k &\rightarrow b^n = (b^2)^k \\
n=2k+1 &\rightarrow b^n = b\times (b^2)^k
\end{aligned}$$

Base on the above observation, we can keep updating $n\leftarrow\lfloor n/2\rfloor$, squaring the base each time $b\leftarrow b^2$, and multiplying by $b$ until $n=0$. This process is the same as calculating the binary representation of $n$ from the right to the left. For example, $12_{10}=1100_2\rightarrow b^{12} = b^{8}\times b^{4}$.

:::

In [ ]:
def fast_pow(base: float, power: int) -> float:
    result = 1

    while power > 0:
        if power % 2 == 1:
            result *= base
        print(f'{result = :<5} {power = :<3} {base = :<4}')
        base *= base
        power //= 2

    return result

fast_pow(2, 12)

### 3.3. Dynamic programming
Dynamic programming (DP) is a method for solving complex problems by breaking them down into simpler overlapping subproblems. It is applicable when the problem has optimal substructure (the optimal solution can be constructed from optimal solutions of its subproblems) and overlapping subproblems (the same subproblems are solved multiple times). DP can be implemented in two main ways:
- Top-down with memoization: This approach uses recursion to solve the problem, storing the results of subproblems in a cache (usually a dictionary) to avoid redundant calculations.
- Bottom-up with tabulation: This approach builds a table (usually an array) iteratively, starting from the smallest subproblems and working up to the original problem.

:::{admonition} Case: Fibonacci numbers
:class: tip

Fibonacci sequence is define as:
- $F(0)=0;\;F(1)=1$
- $F(n)=F(n-1)+F(n-2)$

The naive recursive solution recomputes the same values many times (overlapping subproblems) so it becomes very slow as $n$ grows. Dynamic programming fixes this by storing results and reusing them, making it $O(n)$ time.

:::

In [23]:
adds = 0

def fibo_naive(n):
    global adds
    if n <= 1:
        return n
    result = fibo_naive(n - 1) + fibo_naive(n - 2)
    adds += 1
    return result

fibo_naive(7)
adds

20

In [ ]:
adds = 0

def fib_topdown(n, memo=None):
    global adds
    if memo is None:
        memo = {0: 0, 1: 1}
    if n in memo:
        return memo[n]
    memo[n] = fib_topdown(n - 1, memo) + fib_topdown(n - 2, memo)
    adds += 1
    return memo[n]

fib_topdown(7)
adds

6

In [22]:
adds = 0

def fib_bottomup(n):
    global adds
    if n < 2:
        return n
    a, b = 0, 1
    for _ in range(2, n + 1):
        a, b = b, a + b
        adds += 1
    return b

fib_bottomup(7)
adds

6

:::{admonition} Case: Edit distance
:class: tip

Input: two strings
- $A=a_1a_2\dots a_N$
- $B=b_1b_2\dots b_M$

You are only allowed to insert, delete or substitue a character in $A$. The goal is to find minimum cost to transform $A$ to $B$.

:::

### 3.4. Backtracking
Backtracking is trying choices one by one and once it can't lead to a feasible solution; you undo that choice then try the next option. Intuitively, you can think about solving a maze: when you hit a dead end, you go back to the last junction and try a different path.

:::{admonition} Case: N-Queens
:class: tip

The N-Queens problem challenges placing $N$ chess Queens on an $N\times N$ chess board so that no two Queen attack each other. The for any pair of Queens $i,j$ the constraints can be written as follows:
- Different columns: $x_i\neq x_j$
- Different rows: $y_i\neq y_j$
- Different main diagonals: $x_i-y_i\neq x_j-y_j$
- Different anti-diagonals: $x_i+y_i\neq x_j+y_j$

:::

In [ ]:
def n_queens(n=4):
    cols = set(); diag = set(); anti = set()
    pos = [-1] * n
    result = []

    def backtrack(r):
        if r == n:
            result.append([(i, pos[i]) for i in range(n)])
        
        for c in range(n):
            if c in cols or (r-c) in diag or (r+c) in anti:
                continue

            cols.add(c); diag.add(r-c); anti.add(r+c)
            pos[r] = c
            backtrack(r+1)
            cols.remove(c); diag.remove(r-c); anti.remove(r+c)
            pos[r] = -1
    
    backtrack(0)
    return result

#### Branch and bound
Branch and bound is a backtracking variant for optimization problems. It prunes a branch by proving it cannot be better than the best solution so far.

:::{admonition} Case: 0/1 knapsack
:class: tip

The knapsack problem gives you $N$ items, where the items $n$ has weight $w_n$ and value $v_n$. You need to pick a subset of items with max total value and the total weight is at most $W$.

Backtracking allows you to find all *feasible* subsets by pruning overweighted ones. Branch and bound adds a second, stronger prune: with the remaining capacity, even if the partial choice can't beat the best solution so far, stop.

:::

In [ ]:
def knap_bb(w, v, W):
    n = len(w)
    order = sorted(range(n), key=lambda i: v[i] / w[i], reverse=True)
    best, best_take = 0, [0] * n
    take = [0] * n

    def ub(k, cw, cv):  # fractional upper bound
        cap, val = W - cw, cv
        for t in range(k, n):
            i = order[t]
            if w[i] <= cap:
                cap -= w[i]; val += v[i]
            else:
                val += cap * (v[i] / w[i])
                break
        return val

    def dfs(k, cw, cv):
        nonlocal best, best_take
        if ub(k, cw, cv) <= best: return
        if k == n:
            if cv > best: best, best_take = cv, take.copy()
            return
        i = order[k]
        if cw + w[i] <= W:
            take[i] = 1; dfs(k + 1, cw + w[i], cv + v[i])
        take[i] = 0; dfs(k + 1, cw, cv)

    dfs(0, 0, 0)
    return best, [i for i in range(n) if best_take[i]]

# example:
# print(knap_bb([2,3,4,5], [3,4,5,6], 5))  # (7, [0,1])


## 4. Trade-off methods
- Greedy
- Approximation
- Randomized
- Heuristic
- Online

# Resources
- algorithm-visualizer.org - [Algorithm Visualizer](https://algorithm-visualizer.org/brute-force/pagerank)
- visualgo.net - [VisuAlgo](https://visualgo.net/en/)
- github.com - [The Algorithms - Python](https://github.com/TheAlgorithms/Python/tree/master)